# Projet : Uniform Convergence and the Limits of Learning
**M2 Ingénierie Mathématique et Data Science**

Ce notebook contient l'ensemble des simulations demandées pour étudier la convergence uniforme, le principe du No-Free-Lunch et l'impact du biais inductif.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuration globale des graphiques
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## Partie I : Simulation de la Convergence Uniforme

### 1. Risque Empirique comme Variable Aléatoire

In [ ]:
def task1_empirical_risk():
    p_j = 0.3 # Vrai risque fixe pour une hypothèse h_j
    m_values = [10, 50, 200, 1000]
    n_sim = 10000

    plt.figure(figsize=(12, 8))
    for i, m in enumerate(m_values, 1):
        # Simulation du nombre d'erreurs (Loi Binomiale)
        E_j = np.random.binomial(m, p_j, n_sim)
        L_S = E_j / m # Risque empirique
        
        plt.subplot(2, 2, i)
        plt.hist(L_S, bins=30, density=True, alpha=0.7, color='blue')
        plt.axvline(p_j, color='red', linestyle='dashed', linewidth=2, label=f'True Risk = {p_j}')
        plt.title(f'm = {m}')
        plt.legend()
    
    plt.tight_layout()
    plt.show()

task1_empirical_risk()

**Interprétation :** Conformément à la loi des grands nombres, la variance du risque empirique $L_S(h_j)$ diminue fortement à mesure que la taille de l'échantillon $m$ augmente. Il se concentre autour du vrai risque $L_D(h_j) = 0.3$.

### 2. Déviation entre Risque Empirique et Vrai Risque

In [ ]:
def task2_hoeffding():
    p_j = 0.3
    epsilon = 0.05
    m_values = np.arange(10, 1001, 50)
    n_sim = 5000
    
    prob_exp = []
    prob_hoeffding = []
    
    for m in m_values:
        E_j = np.random.binomial(m, p_j, n_sim)
        L_S = E_j / m
        
        # Probabilité expérimentale de dépassement de epsilon
        prob = np.mean(np.abs(L_S - p_j) > epsilon)
        prob_exp.append(prob)
        
        # Borne théorique de Hoeffding
        bound = 2 * np.exp(-2 * m * epsilon**2)
        prob_hoeffding.append(min(bound, 1.0))
        
    plt.figure(figsize=(10, 6))
    plt.plot(m_values, prob_exp, label='Expérimental', marker='o')
    plt.plot(m_values, prob_hoeffding, label='Borne de Hoeffding', linestyle='--')
    plt.xlabel("Taille de l'échantillon (m)")
    plt.ylabel(f'Pr(|L_S - L_D| > {epsilon})')
    plt.title('Déviation vs Borne de Hoeffding')
    plt.legend()
    plt.show()

task2_hoeffding()

**Interprétation :** La borne de Hoeffding encadre parfaitement la probabilité expérimentale. Cependant, on observe qu'elle est *conservatrice* (elle est souvent beaucoup plus large que la probabilité réelle de déviation observée en pratique).

### 3, 4 et 5. Convergence Uniforme et Taille de la Classe d'Hypothèses

In [ ]:
def task3_4_uniform_convergence():
    epsilon = 0.05
    m_values = np.arange(50, 2001, 150)
    H_sizes = [5, 10, 50, 100, 500] # Différentes tailles de la classe d'hypothèses
    n_sim = 1000
    
    plt.figure(figsize=(12, 7))
    
    for H_size in H_sizes:
        # Assigner des vrais risques aléatoires pour chaque hypothèse de la classe
        p_true = np.random.uniform(0.1, 0.9, H_size)
        prob_exp = []
        
        for m in m_values:
            # Matrice des erreurs : n_sim simulations pour H_size hypothèses
            E_matrix = np.random.binomial(m, p_true, (n_sim, H_size))
            L_S_matrix = E_matrix / m
            
            # Pire déviation parmi toutes les hypothèses (sup_h)
            delta_S = np.max(np.abs(L_S_matrix - p_true), axis=1)
            prob = np.mean(delta_S > epsilon)
            prob_exp.append(prob)
            
        plt.plot(m_values, prob_exp, label=f'|H| = {H_size}', marker='o')
        
    plt.xlabel("Taille de l'échantillon (m)")
    plt.ylabel(f'Pr(Delta_S > {epsilon})')
    plt.title('Impact de la taille de la classe d\'hypothèses sur la convergence uniforme')
    plt.legend()
    plt.show()

task3_4_uniform_convergence()

**Interprétation :** L'augmentation de la taille de la classe d'hypothèses $|\mathcal{H}|$ retarde la convergence uniforme. Plus on a d'hypothèses, plus il est probable qu'au moins l'une d'entre elles dévie fortement de son risque réel par pur hasard statistique. Cela explique la dépendance en $\log|\mathcal{H}|$ dans les bornes théoriques de complexité échantillonnale.

## Partie II : Simulation du No-Free-Lunch

### 6, 7 et 8. Évaluation sur les entrées non vues avec des cibles aléatoires

In [ ]:
def nfl_simulation():
    N = 1000 # Taille de l'espace X
    m_values = np.arange(10, 2001, 200)
    n_sim = 100
    
    exp_unseen_fraction = []
    theo_unseen_fraction = []
    exp_risk = []
    theo_risk = []
    
    for m in m_values:
        unseen_fracs = []
        risks = []
        
        for _ in range(n_sim):
            # On tire m points aléatoirement dans le domaine (avec remise)
            S = np.random.randint(1, N+1, m)
            seen_elements = set(S)
            
            # Calcul des fractions
            unseen_frac = (N - len(seen_elements)) / N
            unseen_fracs.append(unseen_frac)
            
            # Risque vrai = l'algorithme mémorise le vu (0 erreur), 
            # mais sur les points non vus, la fonction cible étant aléatoire Ber(1/2),
            # l'erreur moyenne sur ces points est exactement de 0.5
            risks.append(0.5 * unseen_frac)
            
        exp_unseen_fraction.append(np.mean(unseen_fracs))
        theo_unseen_fraction.append((1 - 1/N)**m)
        
        exp_risk.append(np.mean(risks))
        theo_risk.append(0.5 * (1 - 1/N)**m)
        
    plt.figure(figsize=(10, 6))
    plt.plot(m_values, exp_risk, label='Risque Expérimental', marker='o')
    plt.plot(m_values, theo_risk, label='Risque Théorique E[L_D]', linestyle='--')
    plt.xlabel('Taille de l\'échantillon d\'entraînement (m)')
    plt.ylabel('Risque Vrai Global')
    plt.title('Principe No-Free-Lunch : Apprentissage = Mémorisation pure')
    plt.legend()
    plt.show()

nfl_simulation()

**Interprétation :** Face à une fonction cible sans aucune structure (complètement aléatoire), l'algorithme ne peut faire mieux que deviner avec $50\%$ de succès sur les points qu'il n'a jamais observés. Ainsi, aucune stratégie de prédiction pour les points non vus (prédire tout à 0, tout à 1, ou au hasard) ne surpasse les autres. Il n'y a pas d'apprentissage généralisable possible sans hypothèse préalable.

## Partie III : Introduction de Connaissances a priori (Biais Inductif)

### 9 et 10. Cibles structurées vs Cibles aléatoires

In [ ]:
def inductive_bias_simulation():
    N = 100
    m_values = np.arange(5, 51, 5)
    n_sim = 200
    
    risk_structured = []
    
    for m in m_values:
        risks = []
        for _ in range(n_sim):
            # Création d'une fonction cible structurée (un seuil au hasard)
            theta = np.random.randint(1, N)
            S_x = np.random.randint(1, N+1, m)
            S_y = (S_x > theta).astype(int)
            
            # Apprentissage ERM dans la classe des fonctions seuils
            theta_hat = 0
            for t in range(1, N):
                err = np.mean((S_x > t) != S_y)
                if err == 0:
                    theta_hat = t
                    break
            
            # Évaluation du risque vrai sur tout le domaine X
            domain = np.arange(1, N+1)
            true_y = (domain > theta).astype(int)
            pred_y = (domain > theta_hat).astype(int)
            risks.append(np.mean(true_y != pred_y))
            
        risk_structured.append(np.mean(risks))
        
    # Comparatif avec le risque sous conditions purement aléatoires (NFL)
    risk_random = [0.5 * (1 - 1/N)**m for m in m_values]

    plt.figure(figsize=(10, 6))
    plt.plot(m_values, risk_structured, label='Cibles Structurées (avec Biais Inductif)', marker='o', color='green')
    plt.plot(m_values, risk_random, label='Cibles Aléatoires (NFL)', linestyle='--', color='red')
    plt.xlabel('m (Taille de l\'échantillon)')
    plt.ylabel('Risque Vrai E[L_D]')
    plt.title('Généralisation grâce au Biais Inductif')
    plt.legend()
    plt.show()

inductive_bias_simulation()

## Conclusion (Partie 11)

À travers ces expériences, nous avons mis en évidence plusieurs concepts fondamentaux du Machine Learning :

1. **Concentration autour du vrai risque** : Lorsque la taille de l'échantillon $m$ augmente, le risque empirique devient un estimateur de plus en plus fiable du risque vrai, conformément aux inégalités de concentration (Hoeffding).
2. **Le prix de la dimension ($|\mathcal{H}|$ )** : La convergence uniforme exige de garantir que le risque empirique est proche du risque vrai pour *l'ensemble* des hypothèses simultanément. Plus notre espace de recherche (la classe d'hypothèses) est grand, plus la probabilité de sélectionner une hypothèse qui "surapprend" (overfit) par pur hasard augmente. C'est pourquoi un $|\mathcal{H}|$ plus grand nécessite davantage de données pour converger.
3. **Conservatisme des bornes** : Les bornes théoriques comme celle de Hoeffding assurent une garantie absolue (au pire cas) pour des classes d'hypothèses arbitraires. En pratique, l'algorithme se comporte souvent bien mieux, ce qui rend ces bornes très conservatrices par rapport au comportement empirique moyen.
4. **No-Free-Lunch et prévisibilité** : Lorsque la fonction cible est totalement aléatoire, observer des exemples passés n'apporte strictement aucune information sur le comportement futur de la fonction. L'apprentissage se résume alors à une simple mémorisation, et la généralisation est impossible.
5. **Le pouvoir du biais inductif** : À l'inverse, lorsqu'on introduit une hypothèse a priori (un biais inductif, par exemple l'hypothèse que la cible est une fonction seuil) et que cette hypothèse concorde avec la réalité, l'algorithme est capable d'extrapoler l'information des points observés pour prédire avec justesse les points non observés. C'est cette restriction de la classe de fonctions qui rend la véritable généralisation possible en Data Science.